# Sesión adicional · El ejercicio del laboratorio 7 resuelto
**Una herramienta MCP nueva (`consultar_tasa`) y un segundo agente A2A (Agente de Requisitos) · Notebook resuelto (Colab o local)**

El notebook [`07_MCP_A2A_LangGraph_Gradio.ipynb`](./07_MCP_A2A_LangGraph_Gradio.ipynb) termina con este ejercicio:

> 1. Agrega al servidor MCP una herramienta `consultar_tasa()` que devuelva la tasa ficticia. **Reinicia el entorno** y re-ejecuta: el Asesor la descubre sola.
> 2. Crea un **Agente de Requisitos** A2A en el puerto `9998` (copia el paso 7) y conéctalo al Asesor con una nueva herramienta.

Este notebook es esa sesión, **completa y autocontenida**: vuelve a levantar todas las piezas (no necesitas ejecutar antes el 07) y marca con 🆕 lo que cambia.

| Pieza | Qué es | Protocolo | Puerto |
|---|---|---|---|
| 🧑‍💼 **Asesor** | Agente principal. Ahora con **5 herramientas** | — | — |
| 🗄️ **Servidor MCP "Core bancario"** | `consultar_saldo`, `listar_movimientos` y 🆕 `consultar_tasa` | **MCP** | 8001 |
| 🤖 **Agente de Créditos** | El mismo de la sesión anterior: simula cuotas | **A2A** | 9999 |
| 📋 🆕 **Agente de Requisitos** | **Otro agente**, con su propio LLM y su herramienta `verificar_requisitos` | **A2A** | 9998 |

> ⚠️ Todo es ficticio: no hay dinero real, ni aprobación de créditos, ni datos personales.
> 🔑 Necesitas `GROQ_API_KEY`: en tu PC, en el archivo `.env` de la raíz del repositorio; en Colab, en **Secrets** (icono 🔑). Nunca la pegues en el código.
> 🔁 Si el notebook 07 sigue abierto, **reinicia o cierra su kernel** antes de empezar: sus servidores ocupan los puertos 8001 y 9999.

## 🧠 La idea en una imagen
```
                         ┌──────────────────────────┐
        Cliente ───────► │   🧑‍💼 ASESOR (agente)     │
                         │   LLM decide qué usar    │
                         └───┬──────────┬────────┬──┘
                    MCP      │      A2A │        │ A2A 🆕
                             ▼          ▼        ▼
          ┌───────────────────┐ ┌──────────────┐ ┌─────────────────────┐
          │ 🗄️ Servidor MCP    │ │ 🤖 Agente de  │ │ 📋 Agente de         │
          │ consultar_saldo   │ │   Créditos   │ │   Requisitos 🆕      │
          │ listar_movimient. │ │ simular_cuota│ │ verificar_requisitos│
          │ consultar_tasa 🆕 │ │  (puerto     │ │  (puerto 9998)      │
          │  (puerto 8001)    │ │   9999)      │ │                     │
          └───────────────────┘ └──────────────┘ └─────────────────────┘
```

### Las dos ideas que pone a prueba el ejercicio
| Parte | Lo que cambia | Lo que **no** cambia | Idea |
|---|---|---|---|
| 1 · MCP | Una función nueva en el **servidor** | El código del Asesor (`crear_asesor`) | **Descubrir en vez de declarar**: el Asesor pide `tools/list` y la herramienta aparece sola |
| 2 · A2A | Un servidor A2A nuevo + **una** herramienta en el Asesor | El protocolo, el ejecutor y las rutas: son los del paso 7 | **Delegar es una herramienta más**; agregar un colega = su tarjeta + su número en los contactos |

### La pregunta de referencia
«Gano 3.500.000 al mes y tengo obligaciones por 700.000. ¿Qué tasa manejan? Simula una moto de 20.000.000 con aporte de 5.000.000 a 36 meses y dime si la cuota cabe en los requisitos.»

| Dato | Quién lo calcula | Por dónde viaja | Valor esperado |
|---|---|---|---|
| Tasa | Servidor MCP (`consultar_tasa`) | MCP | 1 % mensual |
| Cuota | Agente de Créditos (`simular_cuota`) | A2A | ≈ $498.215 |
| Límite de cuota | Agente de Requisitos (`verificar_requisitos`) | A2A | $525.000 (35 % × 3,5M − 0,7M) → **la cuota cabe** |

Son las mismas cifras de los laboratorios 5 y 6, así que puedes contrastarlas.

### 🧭 Recorrido
| Paso | Qué haces |
|---|---|
| 0 | Instalar y conectar con Groq |
| 1 | Bitácora compartida |
| **Parte A · MCP** | |
| 2 | Servidor MCP con **3** herramientas (🆕 `consultar_tasa`) |
| 3 | Arrancarlo (🆕 detecta si el puerto lo ocupa otro kernel) |
| 4 | Comprobar el **descubrimiento**: `tools/list` devuelve 3 herramientas |
| **Parte B · A2A** | |
| 5 | 🆕 El paso 7 convertido en una **receta**: `publicar_a2a(...)` |
| 6 | Publicar el Agente de Créditos (puerto 9999) |
| 7 | 🆕 Crear y publicar el **Agente de Requisitos** (puerto 9998) |
| 8 | Descubrir las **dos** Agent Cards y probar al nuevo agente |
| 9 | 🆕 Las llamadas A2A como herramientas del Asesor (ahora dos) |
| **Parte C · Todo junto** | |
| 10 | El Asesor con **5** herramientas y la pregunta de referencia + comprobación |
| 11 | **Gradio**: diagrama de secuencia con 5 carriles |
| 12 | Cerrar la interfaz |

## Paso 0 · ▶️ Instalar y conectar con Groq
Igual que en la sesión anterior: los mismos paquetes y el mismo modelo. No hace falta instalar nada nuevo para el ejercicio.

> Si aparecen avisos rojos de dependencias de paquetes que no usamos (`google-adk`, etc.), puedes ignorarlos.

> **🧩 En palabras simples**
> - **Qué hace esta celda:** instala las librerías que vamos a usar (solo las que falten).
> - **Piénsalo como:** descargar las apps antes de usarlas en el celular.
> - **Qué vas a ver al ejecutarla:** barras de progreso o nada, si ya está todo instalado.

In [ ]:
# Las mismas librerías del notebook 07 (en tu PC, si ya instalaste requirements.txt, no descarga nada).
# gradio, pandas y uvicorn ya vienen en Colab: solo se instalan si faltan.
import importlib.util
MODULOS = {'gradio': 'gradio', 'pandas': 'pandas', 'uvicorn': 'uvicorn'}
faltantes = ' '.join(p for p, m in MODULOS.items() if importlib.util.find_spec(m) is None)
%pip install -q langchain langgraph langchain-groq mcp langchain-mcp-adapters "a2a-sdk[http-server]" python-dotenv {faltantes}

> **🧩 En palabras simples**
> - **Qué hace esta celda:** conecta con el LLM de Groq usando tu clave guardada en `.env` (en tu PC) o en Secrets (en Colab).
> - **Piénsalo como:** enchufar el "cerebro" que van a usar los tres agentes.
> - **Qué vas a ver al ejecutarla:** la palabra `conectado`.

In [ ]:
import os                              # herramientas para leer variables del sistema
from dotenv import load_dotenv         # lee el archivo .env (en tu PC)
from langchain_groq import ChatGroq    # la "conexión" con el LLM de Groq

# 1) Buscamos la clave de Groq SIN escribirla en el código
load_dotenv()                                    # en tu PC: carga el archivo .env si existe
GROQ_API_KEY = os.getenv('GROQ_API_KEY')
if not GROQ_API_KEY:
    try:
        # En Colab: la leemos de Secrets (icono 🔑 a la izquierda)
        from google.colab import userdata
        GROQ_API_KEY = userdata.get('GROQ_API_KEY')
    except ImportError:
        pass                                     # fuera de Colab este módulo no existe
if not GROQ_API_KEY:
    raise ValueError('Configura GROQ_API_KEY en .env (en tu PC) o en Secrets de Colab.')

# 2) El modelo que usarán TODOS los agentes de hoy
MODELO = 'openai/gpt-oss-120b'

# 3) Una "receta" que entrega un LLM listo para usar (cada agente pide el suyo)
def nuevo_llm():
    return ChatGroq(
        model=MODELO,
        api_key=GROQ_API_KEY,
        temperature=0,             # 0 = respuestas lo más estables posible
        reasoning_effort='low',    # que "piense" poco: respuestas más rápidas
    )

# 4) Prueba: le pedimos al LLM que diga "conectado"
print(nuevo_llm().invoke('Responde solo: conectado').content)

## Paso 1 · ▶️ Bitácora compartida
La misma bitácora: cada mensaje entre participantes se anota con `registrar(de, para, protocolo, accion, detalle)`. Ahora hay un participante más: **`Agente Requisitos`**.

> **🧩 En palabras simples**
> - **Qué hace esta celda:** crea el cuaderno de registro donde anotaremos cada mensaje entre participantes.
> - **Piénsalo como:** el libro de minutas de una reunión: quién le dijo qué a quién.
> - **Qué vas a ver al ejecutarla:** `Bitácora lista.`

In [ ]:
import time, json, logging

# Los servidores escriben muchos mensajes técnicos; los silenciamos para ver solo lo importante
for nombre in ['httpx', 'mcp', 'a2a', 'uvicorn']:
    logging.getLogger(nombre).setLevel(logging.WARNING)

# En Windows, al cerrar cada conexión HTTP, asyncio anota un error inofensivo
# (ConnectionResetError, WinError 10054). Ocultamos SOLO ese aviso; los demás errores se siguen viendo.
logging.getLogger('asyncio').addFilter(lambda registro: '_call_connection_lost' not in registro.getMessage())

BITACORA = []

# de / para  → quién le habla a quién · protocolo → 'MCP', 'A2A', 'LLM' o 'Local'
# accion     → qué pidió              · detalle   → los datos que viajaron
def registrar(de, para, protocolo, accion, detalle=''):
    BITACORA.append({
        'hora': time.strftime('%H:%M:%S'),
        'de': de, 'para': para, 'protocolo': protocolo,
        'accion': accion,
        'detalle': detalle if isinstance(detalle, str) else json.dumps(detalle, ensure_ascii=False),
    })

print('Bitácora lista.')

# Parte A · MCP: una herramienta nueva que el Asesor descubre solo

## Paso 2 · ▶️ El servidor MCP con tres herramientas
Las dos herramientas de la sesión anterior quedan **idénticas**. La única novedad es 🆕 `consultar_tasa()`:

| Herramienta | Entrada | Devuelve |
|---|---|---|
| `consultar_saldo` | `cliente_id` | `saldo_cop` |
| `listar_movimientos` | `cliente_id` | `movimientos` y `saldo_final_cop` |
| 🆕 `consultar_tasa` | **nada** | `tasa_mensual` (0,01) y `tasa_mensual_pct` (1,0) |

Fíjate en que `consultar_tasa` **no recibe parámetros**: su esquema JSON de entrada estará vacío. Aun así el LLM sabrá cuándo usarla, porque lee su **docstring**.

Las reglas ficticias del banco quedan en un solo diccionario, `POLITICA_FICTICIA` (como en el laboratorio 5), para que la tasa del servidor MCP y la del Agente de Créditos no puedan diferir.

> **🧩 En palabras simples**
> - **Qué hace esta celda:** crea el servidor MCP con tres herramientas: saldo, movimientos y la nueva, la tasa.
> - **Piénsalo como:** agregar una etiqueta nueva a la caja de herramientas; quien la abra la verá sin que nadie le avise.
> - **Qué vas a ver al ejecutarla:** `Servidor MCP definido con 3 herramientas: ...`.

In [ ]:
from mcp.server.fastmcp import FastMCP   # la librería que convierte funciones en herramientas MCP

# ── Los datos del "core bancario" (inventados) ─────────────────────
CUENTAS = {
    'DEMO-001': {
        'saldo_cop': 5_000_000,
        'movimientos': [
            {'concepto': 'Saldo inicial',    'valor_cop':  4_000_000},
            {'concepto': 'Ingreso ficticio', 'valor_cop':  3_000_000},
            {'concepto': 'Gastos ficticios', 'valor_cop': -2_000_000},
        ],
    }
}

# Reglas inventadas para la clase (las mismas del laboratorio 5; no son de ningún banco real)
POLITICA_FICTICIA = {
    'tasa_mensual': 0.01,        # 1 % mensual
    'fraccion_ingreso': 0.35,    # la cuota no debería pasar del 35 % del ingreso (menos obligaciones)
    'documentos': ['Identificación ficticia', 'Soporte ficticio de ingresos'],
}

servidor_mcp = FastMCP('Core bancario ficticio', log_level='WARNING')

# ── Herramientas de la sesión anterior (sin cambios) ────────────────
@servidor_mcp.tool()
def consultar_saldo(cliente_id: str) -> dict:
    '''Devuelve el saldo disponible (COP) de una cuenta ficticia. Ejemplo de cliente_id: DEMO-001.'''
    registrar('Asesor', 'Servidor MCP', 'MCP', 'tools/call consultar_saldo', {'cliente_id': cliente_id})
    cuenta = CUENTAS.get(cliente_id)
    resultado = {'cliente_id': cliente_id, 'saldo_cop': cuenta['saldo_cop']} if cuenta else {'error': 'cliente no existe'}
    registrar('Servidor MCP', 'Asesor', 'MCP', 'resultado consultar_saldo', resultado)
    return resultado

@servidor_mcp.tool()
def listar_movimientos(cliente_id: str) -> dict:
    '''Devuelve los movimientos y el saldo final de una cuenta ficticia. Ejemplo de cliente_id: DEMO-001.'''
    registrar('Asesor', 'Servidor MCP', 'MCP', 'tools/call listar_movimientos', {'cliente_id': cliente_id})
    cuenta = CUENTAS.get(cliente_id)
    if not cuenta:
        resultado = {'error': 'cliente no existe'}
    else:
        movs = cuenta['movimientos']
        resultado = {'movimientos': movs, 'saldo_final_cop': sum(m['valor_cop'] for m in movs)}
    registrar('Servidor MCP', 'Asesor', 'MCP', 'resultado listar_movimientos', resultado)
    return resultado

# ── 🆕 EJERCICIO 1 · La herramienta nueva ───────────────────────────
# No recibe parámetros: la tasa es la misma para todos los clientes.
# El docstring es lo que el LLM del Asesor leerá para decidir cuándo usarla.
@servidor_mcp.tool()
def consultar_tasa() -> dict:
    '''Devuelve la tasa de interés mensual ficticia que usa el banco para simular créditos.'''
    registrar('Asesor', 'Servidor MCP', 'MCP', 'tools/call consultar_tasa', {})
    tasa = POLITICA_FICTICIA['tasa_mensual']
    resultado = {'tasa_mensual': tasa, 'tasa_mensual_pct': tasa * 100, 'nota': 'tasa ficticia para la clase'}
    registrar('Servidor MCP', 'Asesor', 'MCP', 'resultado consultar_tasa', resultado)
    return resultado

nombres = [h.name for h in await servidor_mcp.list_tools()]
print(f'Servidor MCP definido con {len(nombres)} herramientas: {nombres}')

## Paso 3 · ▶️ Arrancar los servidores en segundo plano
La misma función `arrancar(app, puerto)` del notebook 07, con una mejora 🆕: distingue **dos** casos de "puerto ocupado".

| Caso | Qué significa | Qué hace |
|---|---|---|
| El puerto lo abrió **este** notebook | Re-ejecutaste la celda | Nada (como antes) |
| El puerto lo ocupa **otro** programa | Casi siempre, el kernel del notebook 07 sigue vivo | **Se detiene con un error claro**. Antes seguía en silencio y usaba el servidor viejo, que no tiene `consultar_tasa` |

Ese segundo caso es justo la trampa del ejercicio 1: «**reinicia el entorno** y re-ejecuta».

Los tres puertos quedan juntos en una sola celda, para cambiarlos fácilmente si alguno está ocupado.

> **🧩 En palabras simples**
> - **Qué hace esta celda:** define los puertos, pone a funcionar el servidor MCP en la "ventanilla" 8001 y comprueba que nadie más la esté usando.
> - **Piénsalo como:** abrir una ventanilla, pero antes mirar que no haya otro empleado atendiendo en ella con información vieja.
> - **Qué vas a ver al ejecutarla:** `Servidor escuchando en el puerto 8001`. Si ves `RuntimeError: El puerto 8001 está ocupado…`, reinicia el kernel del notebook 07.

In [ ]:
import threading, socket, uvicorn

# Los tres puertos de hoy (127.0.0.1 = "este mismo computador")
PUERTO_MCP, PUERTO_CREDITOS, PUERTO_REQUISITOS = 8001, 9999, 9998
URL_MCP = f'http://127.0.0.1:{PUERTO_MCP}/mcp'

# Puertos que abrió ESTE kernel (se conserva si re-ejecutas la celda)
PUERTOS_PROPIOS = globals().get('PUERTOS_PROPIOS', set())

def puerto_ocupado(puerto):
    with socket.socket() as s:
        return s.connect_ex(('127.0.0.1', puerto)) == 0

def arrancar(app, puerto):
    if puerto in PUERTOS_PROPIOS:
        print(f'El puerto {puerto} ya lo abrió este notebook (reinicia el kernel si cambiaste su código).')
        return
    if puerto_ocupado(puerto):
        # 🆕 Otro programa (casi siempre, el kernel del notebook 07) ya escucha aquí con su código viejo
        raise RuntimeError(f'El puerto {puerto} está ocupado por otro programa, probablemente el kernel del '
                           'notebook 07. Reinicia o cierra ese kernel y vuelve a ejecutar esta celda.')
    servidor = uvicorn.Server(uvicorn.Config(app, host='127.0.0.1', port=puerto, log_level='warning'))
    threading.Thread(target=servidor.run, daemon=True).start()
    while not puerto_ocupado(puerto):
        time.sleep(0.2)
    PUERTOS_PROPIOS.add(puerto)
    print(f'Servidor escuchando en el puerto {puerto}')

arrancar(servidor_mcp.streamable_http_app(), PUERTO_MCP)

## Paso 4 · ▶️ Comprobar el descubrimiento
Repetimos el paso 4 del notebook 07: `initialize` → `tools/list` → `tools/call`. Esta vez llamamos a la herramienta nueva.

**📤 Salida esperada:** **3** herramientas; `consultar_tasa` con `entrada: {}` (no pide datos) y el resultado `tasa_mensual: 0.01`.

> **🧩 En palabras simples**
> - **Qué hace esta celda:** le pregunta al servidor MCP qué herramientas ofrece y usa la nueva.
> - **Piénsalo como:** volver a la ventanilla y pedir el catálogo actualizado.
> - **Qué vas a ver al ejecutarla:** la lista de 3 herramientas y la tasa `0.01`.

In [ ]:
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client

async with streamable_http_client(URL_MCP) as (leer, escribir, _):
    async with ClientSession(leer, escribir) as sesion:
        info = await sesion.initialize()                          # 1) initialize
        print('1) Servidor:', info.serverInfo.name)

        lista = await sesion.list_tools()                         # 2) tools/list
        print(f'\n2) Herramientas disponibles: {len(lista.tools)}')
        for h in lista.tools:
            nueva = '  🆕' if h.name == 'consultar_tasa' else ''
            print(f'   • {h.name}{nueva}: {h.description}')
            print(f'     entrada: {h.inputSchema.get("properties", {})}')

        r = await sesion.call_tool('consultar_tasa', {})          # 3) tools/call (sin argumentos)
        print('\n3) Resultado de tools/call consultar_tasa:', r.content[0].text)

**🤔 Observa:** no tocamos ningún cliente. Cualquier cliente MCP que se conecte (el Asesor, Claude Desktop, VS Code…) verá la herramienta nueva en su próximo `tools/list`.

# Parte B · A2A: un segundo agente

## Paso 5 · ▶️ El paso 7 convertido en una receta
El ejercicio dice «**copia el paso 7**». Si lo copiamos tal cual, quedan dos bloques casi idénticos. Mejor vemos **qué cambia** de un agente a otro y lo convertimos en parámetros:

| Parte del paso 7 | ¿Cambia entre agentes? | En la receta |
|---|---|---|
| Nombre y descripción de la **Agent Card** | Sí | `nombre`, `descripcion` |
| La *skill* que anuncia | Sí | `skill` |
| El agente de LangGraph que trabaja | Sí | `crear_agente` |
| El puerto | Sí | `puerto` |
| El **ejecutor** (leer el texto → invocar al agente → devolver su respuesta) | **No** | Igual para todos |
| Las **rutas** (`/.well-known/agent-card.json` y `POST /`) | **No** | Igual para todos |

Lo que no cambia **es el protocolo**: por eso cualquier agente A2A se publica igual.

> **🧩 En palabras simples**
> - **Qué hace esta celda:** crea una "receta" para publicar cualquier agente como servidor A2A.
> - **Piénsalo como:** un molde de tarjeta de presentación: cada colega escribe su nombre y su especialidad, pero el formato es el mismo.
> - **Qué vas a ver al ejecutarla:** `Receta publicar_a2a lista.`

In [ ]:
from starlette.applications import Starlette
from a2a.server.agent_execution import AgentExecutor
from a2a.server.request_handlers import DefaultRequestHandler
from a2a.server.tasks import InMemoryTaskStore
from a2a.server.routes import create_agent_card_routes, create_jsonrpc_routes
from a2a.types import AgentCard, AgentInterface, AgentSkill, AgentCapabilities
from a2a.helpers.proto_helpers import new_text_message

# El EJECUTOR del paso 7, ahora sirve para cualquier agente: recibe la receta que lo crea
class EjecutorLangGraph(AgentExecutor):
    def __init__(self, crear_agente):
        self.crear_agente = crear_agente

    async def execute(self, context, event_queue):
        pregunta = context.get_user_input()                            # 1) leemos el texto que llegó
        r = await self.crear_agente().ainvoke({'messages': [('user', pregunta)]})   # 2) el agente trabaja
        await event_queue.enqueue_event(new_text_message(r['messages'][-1].text))   # 3) devolvemos su respuesta

    async def cancel(self, context, event_queue):
        raise Exception('Cancelar no está soportado en este ejemplo')

# La RECETA: Agent Card + ejecutor + rutas + arrancar. Devuelve la dirección del agente.
def publicar_a2a(nombre, descripcion, skill, crear_agente, puerto):
    url = f'http://127.0.0.1:{puerto}/'
    tarjeta = AgentCard(
        name=nombre, description=descripcion, version='1.0.0',
        supported_interfaces=[AgentInterface(url=url, protocol_binding='JSONRPC', protocol_version='1.0')],
        capabilities=AgentCapabilities(streaming=False),
        default_input_modes=['text/plain'], default_output_modes=['text/plain'],
        skills=[skill],
    )
    manejador = DefaultRequestHandler(agent_executor=EjecutorLangGraph(crear_agente),
                                      task_store=InMemoryTaskStore(), agent_card=tarjeta)
    app = Starlette(routes=create_agent_card_routes(tarjeta) + create_jsonrpc_routes(manejador, rpc_url='/'))
    arrancar(app, puerto)
    return url

print('Receta publicar_a2a lista.')

## Paso 6 · ▶️ Publicar el Agente de Créditos
El mismo agente de la sesión anterior (misma herramienta, mismas instrucciones, misma Agent Card), ahora publicado con la receta. La tasa la lee de `POLITICA_FICTICIA`, la misma fuente que usa `consultar_tasa`.

> **🧩 En palabras simples**
> - **Qué hace esta celda:** crea el Agente de Créditos y lo publica en el puerto 9999.
> - **Piénsalo como:** volver a abrir la oficina del colega de créditos, igual que la sesión anterior.
> - **Qué vas a ver al ejecutarla:** `Servidor escuchando en el puerto 9999`.

In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool

# Herramienta LOCAL del Agente de Créditos (nadie de afuera la ve)
@tool
def simular_cuota(precio_cop: int, aporte_cop: int, meses: int) -> dict:
    '''Simula la cuota mensual fija de un crédito ficticio al 1 % mensual.'''
    registrar('Agente Créditos', 'Agente Créditos', 'Local', 'herramienta simular_cuota',
              {'precio': precio_cop, 'aporte': aporte_cop, 'meses': meses})
    tasa = POLITICA_FICTICIA['tasa_mensual']
    capital = precio_cop - aporte_cop
    cuota = capital * tasa / (1 - (1 + tasa) ** (-meses))   # fórmula de cuota fija
    return {'capital_cop': capital, 'meses': meses, 'cuota_aproximada_cop': round(cuota)}

def crear_agente_creditos():
    return create_agent(
        model=nuevo_llm(),
        tools=[simular_cuota],
        system_prompt='Eres el Agente de Créditos de un banco ficticio. Usa simular_cuota para cualquier cálculo. '
                      'Responde en máximo 3 frases, con las cifras de la herramienta. Nunca apruebes créditos.',
    )

URL_CREDITOS = publicar_a2a(
    nombre='Agente de Créditos',
    descripcion='Simula cuotas de créditos ficticios. No aprueba créditos.',
    skill=AgentSkill(id='simular_cuota', name='Simular cuota',
                     description='Calcula la cuota mensual dado precio, aporte y plazo.',
                     tags=['crédito', 'simulación'],
                     examples=['Simula una moto de 20 millones con 5 de aporte a 36 meses']),
    crear_agente=crear_agente_creditos,
    puerto=PUERTO_CREDITOS,
)

## Paso 7 · ▶️ 🆕 Crear y publicar el Agente de Requisitos
**Ejercicio 2.** Un agente **independiente**, con su propio LLM y su propia herramienta local:

| `verificar_requisitos` | |
|---|---|
| Entrada | `ingreso_mensual_cop`, `obligaciones_cop` y, opcional, `cuota_cop` |
| Regla (ficticia, la del laboratorio 5) | Límite de cuota = 35 % del ingreso − obligaciones (nunca negativo) |
| Devuelve | `limite_cuota_ilustrativo_cop`, `documentos` y, si recibió una cuota, `cuota_cabe` |

¿Por qué un **agente** y no otra herramienta MCP? El pedido llega en lenguaje natural («gano 3,5 millones, debo 700 mil…»): alguien tiene que **interpretarlo**, extraer las cifras, llamar a la herramienta y **explicar** el resultado sin aprobar el crédito. Eso es razonar, y para eso tiene su propio LLM.

> **🧩 En palabras simples**
> - **Qué hace esta celda:** crea el Agente de Requisitos con su calculadora y lo publica en el puerto 9998.
> - **Piénsalo como:** contratar a un colega nuevo, darle su propia calculadora y pegar su tarjeta en la puerta 9998.
> - **Qué vas a ver al ejecutarla:** `Servidor escuchando en el puerto 9998`.

In [ ]:
# Herramienta LOCAL del Agente de Requisitos: el Asesor no la ve ni sabe que existe
@tool
def verificar_requisitos(ingreso_mensual_cop: int, obligaciones_cop: int, cuota_cop: int = 0) -> dict:
    '''Calcula el límite ilustrativo de cuota (35 % del ingreso menos obligaciones) y los documentos requeridos.
    Si recibe cuota_cop mayor que 0, indica si esa cuota cabe en el límite. No aprueba créditos.'''
    registrar('Agente Requisitos', 'Agente Requisitos', 'Local', 'herramienta verificar_requisitos',
              {'ingreso': ingreso_mensual_cop, 'obligaciones': obligaciones_cop, 'cuota': cuota_cop})
    limite = max(0, round(ingreso_mensual_cop * POLITICA_FICTICIA['fraccion_ingreso'] - obligaciones_cop))
    resultado = {'limite_cuota_ilustrativo_cop': limite, 'documentos': POLITICA_FICTICIA['documentos']}
    if cuota_cop > 0:
        resultado['cuota_cop'] = cuota_cop
        resultado['cuota_cabe'] = cuota_cop <= limite
    return resultado

def crear_agente_requisitos():
    return create_agent(
        model=nuevo_llm(),
        tools=[verificar_requisitos],
        system_prompt='Eres el Agente de Requisitos de un banco ficticio. Usa verificar_requisitos para cualquier '
                      'cálculo. Responde en máximo 3 frases, con las cifras de la herramienta, e indica los documentos. '
                      'El límite es ilustrativo: nunca apruebes ni niegues créditos.',
    )

URL_REQUISITOS = publicar_a2a(
    nombre='Agente de Requisitos',
    descripcion='Calcula un límite ilustrativo de cuota según ingresos y obligaciones, e indica los documentos. '
                'No aprueba créditos.',
    skill=AgentSkill(id='verificar_requisitos', name='Verificar requisitos',
                     description='Dado el ingreso mensual, las obligaciones y opcionalmente una cuota, '
                                 'calcula el límite de cuota y si la cuota cabe.',
                     tags=['crédito', 'requisitos', 'capacidad de pago'],
                     examples=['Gano 3.500.000, tengo obligaciones por 700.000: ¿cabe una cuota de 498.215?']),
    crear_agente=crear_agente_requisitos,
    puerto=PUERTO_REQUISITOS,
)

## Paso 8 · ▶️ Descubrir a los dos agentes y probar al nuevo
**8.1** Cualquier cliente encuentra a un agente leyendo su Agent Card en la dirección conocida `/.well-known/agent-card.json`. Leemos **las dos**: así se vería un pequeño **directorio de agentes**.

**8.2** Le enviamos un mensaje al Agente de Requisitos, sin pasar por el Asesor.

**📤 Salida esperada:** las dos tarjetas (nombre, *skill*, puerto) y una respuesta con el límite de **$525.000**, que la cuota de 498.215 **cabe** y los dos documentos.

> **🧩 En palabras simples**
> - **Qué hace esta celda:** lee las tarjetas de los dos agentes y le hace una pregunta directa al nuevo.
> - **Piénsalo como:** leer las tarjetas pegadas en dos puertas y luego pasarle una nota al colega nuevo.
> - **Qué vas a ver al ejecutarla:** dos tarjetas y la respuesta del Agente de Requisitos.

In [ ]:
import httpx, uuid
from a2a.client import create_client, ClientConfig
from a2a.types import Message, Part, Role, SendMessageRequest

# ── 8.1 DESCUBRIR: el "directorio" de agentes ──
for url in [URL_CREDITOS, URL_REQUISITOS]:
    card = httpx.get(url + '.well-known/agent-card.json').json()
    skill = card['skills'][0]
    print(f"📇 {card['name']}  ({card['supportedInterfaces'][0]['url']})")
    print(f"   {card['description']}")
    print(f"   skill: {skill['id']} · ejemplo: {skill['examples'][0]}\n")

# ── 8.2 ENVIAR: la receta del paso 9 del notebook 07, ahora con la dirección como parámetro ──
async def enviar_a2a(url, texto):
    async with httpx.AsyncClient(timeout=120) as http:
        cliente = await create_client(url, client_config=ClientConfig(streaming=False, httpx_client=http))
        pedido = SendMessageRequest(message=Message(
            role=Role.ROLE_USER, message_id=str(uuid.uuid4()), parts=[Part(text=texto)]))
        async for evento in cliente.send_message(pedido):
            if evento.HasField('message'):
                return ' '.join(p.text for p in evento.message.parts)
    return '(sin respuesta)'

print('💬 Agente de Requisitos:',
      await enviar_a2a(URL_REQUISITOS, 'Gano 3.500.000 al mes y tengo obligaciones por 700.000. '
                                       '¿Cabe una cuota de 498.215?'))

## Paso 9 · ▶️ 🆕 Las llamadas A2A como herramientas del Asesor
Una herramienta por agente. El **docstring** es la interfaz para el LLM del Asesor: le dice **cuándo** delegar en cada uno y **qué datos** incluir en el pedido.

| Herramienta | Delega en | El docstring le pide incluir |
|---|---|---|
| `consultar_agente_creditos` | Agente de Créditos (9999) | precio, aporte y plazo |
| 🆕 `consultar_agente_requisitos` | Agente de Requisitos (9998) | ingreso, obligaciones y, si ya la tiene, la cuota |

> **🧩 En palabras simples**
> - **Qué hace esta celda:** agrega al colega nuevo a los contactos del Asesor.
> - **Piénsalo como:** guardar un número nuevo en el celular, con una nota de para qué llamarlo.
> - **Qué vas a ver al ejecutarla:** `Herramientas A2A del Asesor: ['consultar_agente_creditos', 'consultar_agente_requisitos']`.

In [ ]:
@tool
async def consultar_agente_creditos(pedido: str) -> str:
    '''Delega al Agente de Créditos (vía A2A) cualquier simulación de crédito o cuota.
    Incluye en el pedido el precio, el aporte y el plazo en meses.'''
    registrar('Asesor', 'Agente Créditos', 'A2A', 'SendMessage', pedido)
    respuesta = await enviar_a2a(URL_CREDITOS, pedido)
    registrar('Agente Créditos', 'Asesor', 'A2A', 'Message (respuesta)', respuesta)
    return respuesta

# 🆕 EJERCICIO 2 · La herramienta que conecta al Asesor con el Agente de Requisitos
@tool
async def consultar_agente_requisitos(pedido: str) -> str:
    '''Delega al Agente de Requisitos (vía A2A) la verificación de requisitos o capacidad de pago:
    el límite ilustrativo de cuota según ingresos y obligaciones, y los documentos requeridos.
    Incluye en el pedido el ingreso mensual, las obligaciones y, si ya la tienes, la cuota a comparar.'''
    registrar('Asesor', 'Agente Requisitos', 'A2A', 'SendMessage', pedido)
    respuesta = await enviar_a2a(URL_REQUISITOS, pedido)
    registrar('Agente Requisitos', 'Asesor', 'A2A', 'Message (respuesta)', respuesta)
    return respuesta

HERRAMIENTAS_A2A = [consultar_agente_creditos, consultar_agente_requisitos]
print('Herramientas A2A del Asesor:', [h.name for h in HERRAMIENTAS_A2A])

# Parte C · Todo junto

## Paso 10 · ▶️ El Asesor con cinco herramientas
`crear_asesor()` es **la misma receta** del notebook 07: pide las herramientas al servidor MCP con `get_tools()` y les suma las A2A. Solo cambian dos cosas:

1. La lista A2A ahora tiene dos herramientas (`HERRAMIENTAS_A2A`).
2. Las instrucciones explican **el orden**: para saber si una cuota cabe, primero hay que pedir la cuota y después mandarla a Requisitos.

Para `consultar_tasa` **no escribimos nada**: llega sola con `get_tools()`.

| Herramienta | Protocolo | ¿Quién responde? |
|---|---|---|
| `consultar_saldo`, `listar_movimientos` | MCP | Servidor MCP (función) |
| 🆕 `consultar_tasa` | MCP | Servidor MCP (función), **descubierta**, no declarada |
| `consultar_agente_creditos` | A2A | Agente de Créditos (otro LLM) |
| 🆕 `consultar_agente_requisitos` | A2A | Agente de Requisitos (otro LLM) |

**📤 Salida esperada:** en la bitácora, `consultar_tasa` por **MCP**, una delegación **A2A** a cada agente con su herramienta **local** dentro y, al final, una comprobación con ✅.

> **🧩 En palabras simples**
> - **Qué hace esta celda:** crea al Asesor, le hace la pregunta de referencia y comprueba en la bitácora que cada pieza participó.
> - **Piénsalo como:** el asesor consulta el sistema (MCP), llama al colega de créditos y luego al de requisitos (A2A).
> - **Qué vas a ver al ejecutarla:** las 5 herramientas, la tabla de mensajes, la respuesta final y una lista de comprobaciones ✅.

In [ ]:
import re
import pandas as pd
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.agents.middleware import wrap_tool_call

# Ajuste técnico (igual que en el notebook 07): MCP devuelve "bloques" y Groq necesita texto simple
@wrap_tool_call
async def resultado_como_texto(request, handler):
    mensaje = await handler(request)
    mensaje.content = mensaje.text
    return mensaje

cliente_mcp = MultiServerMCPClient({'core_bancario': {'transport': 'streamable_http', 'url': URL_MCP}})

async def crear_asesor():
    # Las herramientas MCP se DESCUBREN (tools/list): aquí no aparece el nombre consultar_tasa
    herramientas = await cliente_mcp.get_tools() + HERRAMIENTAS_A2A
    return create_agent(
        model=nuevo_llm(),
        tools=herramientas,
        middleware=[resultado_como_texto],
        system_prompt=('Eres el Asesor de un banco ficticio. El cliente es DEMO-001. '
                       'Para saldo, movimientos o la tasa usa las herramientas MCP. '
                       'Para simular créditos o cuotas delega con consultar_agente_creditos. '
                       'Para requisitos, capacidad de pago o documentos delega con consultar_agente_requisitos; '
                       'si debes comparar una cuota con los requisitos, primero obtén la cuota y luego envíala. '
                       'No inventes cifras. Responde en español, breve. No apruebes créditos.'),
    )

asesor = await crear_asesor()
print('Herramientas del Asesor:', [h.name for h in (await cliente_mcp.get_tools()) + HERRAMIENTAS_A2A])

PREGUNTA = ('Gano 3.500.000 al mes y tengo obligaciones por 700.000. ¿Qué tasa manejan? '
            'Simula una moto de 20.000.000 con aporte de 5.000.000 a 36 meses y dime si la cuota cabe en los requisitos.')

BITACORA.clear()
registrar('Usuario', 'Asesor', 'LLM', 'pregunta', PREGUNTA)
final = await asesor.ainvoke({'messages': [('user', PREGUNTA)]})
respuesta = final['messages'][-1].text
registrar('Asesor', 'Usuario', 'LLM', 'respuesta', respuesta)

with pd.option_context('display.max_colwidth', 90):
    display(pd.DataFrame(BITACORA)[['de', 'para', 'protocolo', 'accion', 'detalle']])
print('\n💬', respuesta)

# ── Comprobación: ¿participó cada pieza? (la decide el LLM, por eso se revisa y no se supone) ──
def hubo(protocolo, accion, para=None):
    return any(e['protocolo'] == protocolo and e['accion'] == accion and (para is None or e['para'] == para)
               for e in BITACORA)

cifras = re.sub(r'\D', '', respuesta)   # solo los dígitos de la respuesta: "$498.215" → "498215"
comprobaciones = {
    'Ejercicio 1 · el Asesor usó consultar_tasa (MCP)':          hubo('MCP', 'tools/call consultar_tasa'),
    'Delegó en el Agente de Créditos (A2A)':                     hubo('A2A', 'SendMessage', 'Agente Créditos'),
    'Ejercicio 2 · delegó en el Agente de Requisitos (A2A)':     hubo('A2A', 'SendMessage', 'Agente Requisitos'),
    'El Agente de Requisitos usó su herramienta local':          hubo('Local', 'herramienta verificar_requisitos'),
    'La respuesta menciona la cuota (498.215)':                  '498215' in cifras,
    'La respuesta menciona el límite (525.000)':                 '525000' in cifras,
}
print()
for texto, ok in comprobaciones.items():
    print('✅' if ok else '⚠️ ', texto)
if not all(comprobaciones.values()):
    print('\n⚠️  El LLM decidió distinto esta vez: revisa la bitácora o repite la celda con la pregunta más explícita.')

**🤔 Observa en la bitácora:**
- `tools/call consultar_tasa` aparece aunque **nadie** escribió ese nombre en `crear_asesor()`.
- Las delegaciones a Créditos y a Requisitos suelen ir **en orden**: la cuota que devuelve Créditos viaja en el pedido a Requisitos. Si el LLM las pide **a la vez**, Requisitos solo puede dar el límite, y la comparación la hace el Asesor.
- El Asesor **no ve** `simular_cuota` ni `verificar_requisitos`: solo recibe el texto que redactó cada agente.

## Paso 11 · ▶️ Gradio: diagrama de secuencia con cinco carriles
La interfaz del notebook 07 con un carril más (📋 **Agente Requisitos**) y nuevos ejemplos. Los colores no cambian:

| Color | Protocolo |
|---|---|
| 🟦 azul | Usuario ↔ Asesor |
| 🟩 verde | **MCP** (herramientas) |
| 🟪 morado | **A2A** (agente ↔ agente) |
| 🟧 naranja | Herramienta **local** de un agente remoto |

> **🧩 En palabras simples**
> - **Qué hace esta celda:** crea la interfaz visual con el diagrama de flechas, ahora con cinco participantes.
> - **Piénsalo como:** una pantalla que dibuja en vivo quién le habla a quién.
> - **Qué vas a ver al ejecutarla:** la interfaz (en tu PC, en `http://127.0.0.1:7860`; en Colab, además, un enlace público).

In [ ]:
# ═════════════════════════════════════════════════════════════════════
# Igual que el paso 11 del notebook 07. Cambian solo:
#  - CARRILES / ICONOS: un participante más (Agente Requisitos)
#  - la cuadrícula usa len(CARRILES) columnas en vez de 4 fijas
#  - los ejemplos de la interfaz
# ═════════════════════════════════════════════════════════════════════
import asyncio
import gradio as gr

CARRILES = ['Usuario', 'Asesor', 'Servidor MCP', 'Agente Créditos', 'Agente Requisitos']
ICONOS   = {'Usuario': '👤', 'Asesor': '🧑‍💼', 'Servidor MCP': '🗄️', 'Agente Créditos': '🤖', 'Agente Requisitos': '📋'}
COLORES  = {'LLM': '#2563eb', 'MCP': '#0d9488', 'A2A': '#7c3aed', 'Local': '#d97706'}

def dibujar_secuencia(eventos):
    cab = ''.join(f'<div style="text-align:center;font-weight:700;color:#0f172a;padding:8px;'
                  f'background:#e2e8f0;border-radius:8px">{ICONOS[c]} {c}</div>' for c in CARRILES)
    filas = ''
    for i, ev in enumerate(eventos, start=1):
        a, b = CARRILES.index(ev['de']), CARRILES.index(ev['para'])
        color = COLORES[ev['protocolo']]
        izq, der = min(a, b), max(a, b)
        etiqueta = f'{i}. [{ev["protocolo"]}] {ev["accion"]}'
        if a == b:   # mensaje a sí mismo (herramienta local)
            flecha = f'<div style="border:2px dashed {color};border-radius:8px;padding:2px;text-align:center;color:{color};font-weight:700">↻ trabaja por dentro</div>'
        else:
            punta = '▶' if b > a else '◀'
            linea = f'<div style="flex:1;height:3px;background:{color}"></div>'
            flecha = (f'<div style="display:flex;align-items:center;color:{color}">'
                      + (linea + punta if b > a else punta + linea) + '</div>')
        filas += (f'<div style="grid-column:{izq + 1}/{der + 2};padding:2px 10%">'
                  f'<div style="font-size:12px;color:{color};font-weight:600;text-align:center">{etiqueta}</div>'
                  f'{flecha}</div>')
    return (f'<div style="background:#ffffff;border-radius:12px;padding:12px;color:#0f172a">'
            f'<div style="display:grid;grid-template-columns:repeat({len(CARRILES)},1fr);gap:6px 0">{cab}{filas}</div>'
            f'{"" if eventos else "<p style=color:#64748b>Sin mensajes todavía</p>"}</div>')

def tabla(eventos):
    return [[i, e['de'], e['para'], e['protocolo'], e['accion'], e['detalle'][:160]]
            for i, e in enumerate(eventos, start=1)]

async def atender(pregunta):
    BITACORA.clear()
    registrar('Usuario', 'Asesor', 'LLM', 'pregunta', pregunta)
    yield dibujar_secuencia(BITACORA), tabla(BITACORA), '⏳ Pensando…'

    asesor = await crear_asesor()
    trabajo = asyncio.create_task(asesor.ainvoke({'messages': [('user', pregunta)]}))
    while not trabajo.done():
        await asyncio.sleep(0.4)
        yield dibujar_secuencia(BITACORA), tabla(BITACORA), '⏳ Pensando…'

    try:
        respuesta = trabajo.result()['messages'][-1].text
    except Exception as error:
        respuesta = f'⛔ Error: {error}'
    registrar('Asesor', 'Usuario', 'LLM', 'respuesta', respuesta)
    yield dibujar_secuencia(BITACORA), tabla(BITACORA), respuesta

with gr.Blocks(title='MCP y A2A · sesión adicional') as demo:
    gr.Markdown('# 🏦 MCP y A2A · sesión adicional\n🟩 **MCP** = herramientas · 🟪 **A2A** = otro agente · '
                '🟧 herramienta local del agente remoto. Datos ficticios.')
    pregunta = gr.Textbox(label='Pregunta del cliente', value='¿Qué tasa manejan?')
    gr.Examples([['¿Qué tasa manejan?'],
                 ['Gano 3.500.000 y tengo obligaciones por 700.000: ¿qué cuota máxima puedo pagar y qué documentos necesito?'],
                 ['Simula una moto de 20.000.000 con aporte de 5.000.000 a 36 meses'],
                 [PREGUNTA]],
                inputs=pregunta)
    boton = gr.Button('Enviar', variant='primary')
    secuencia = gr.HTML()
    respuesta = gr.Textbox(label='Respuesta del Asesor', lines=4)
    detalle = gr.Dataframe(headers=['#', 'De', 'Para', 'Protocolo', 'Acción', 'Datos'],
                           wrap=True, interactive=False, label='Mensajes (qué viajó)',
                           column_widths=['5%', '12%', '12%', '9%', '20%', '42%'])
    boton.click(atender, pregunta, [secuencia, detalle, respuesta])

try:
    import google.colab
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

# share=EN_COLAB → enlace público solo en Colab; en tu PC la interfaz solo se abre en tu equipo
demo.launch(share=EN_COLAB, debug=False)

## Paso 12 · ▶️ Cerrar la interfaz

> **🧩 En palabras simples**
> - **Qué hace esta celda:** cierra la interfaz.
> - **Piénsalo como:** cerrar la ventanilla al final del día.
> - **Qué vas a ver al ejecutarla:** nada, o un mensaje de cierre.

In [ ]:
# Apaga el servidor de Gradio (libera el puerto y, en Colab, el enlace público)
demo.close()

## 🎯 Recorrido guiado en la interfaz
| # | Pregunta | Qué observar |
|---|---|---|
| 1 | «¿Qué tasa manejan?» | Solo flechas **verdes**: `consultar_tasa`, la herramienta que el Asesor descubrió |
| 2 | «Gano 3.500.000… ¿qué cuota máxima puedo pagar?» | Flecha **morada** solo al 📋 Agente de Requisitos y, dentro, ↻ **naranja** |
| 3 | «Simula una moto…» | Flecha **morada** al 🤖 Agente de Créditos, como en la sesión anterior |
| 4 | La pregunta de referencia | MCP + **dos** delegaciones A2A. Mira si Requisitos recibe la cuota en su pedido (orden) o solo los ingresos (paralelo) |

## 🔍 Qué cambió respecto al notebook 07
| Pieza | Cambio | ¿Hubo que tocar el Asesor? |
|---|---|---|
| Servidor MCP | +1 función con `@servidor_mcp.tool()` | **No**: la descubre con `tools/list` |
| Agente de Requisitos | Servidor A2A nuevo (puerto 9998) con su Agent Card | — |
| Herramientas del Asesor | +1 `@tool` que envuelve la llamada A2A | **Sí, una línea**: la lista `HERRAMIENTAS_A2A` y una frase en sus instrucciones |
| `arrancar` | Detecta si el puerto lo ocupa otro kernel | — |
| Gradio | +1 carril | — |

## 🤔 Preguntas para discutir
1. Agregar una herramienta MCP no tocó al Asesor; agregar un agente A2A sí (una herramienta). ¿Por qué esa diferencia? ¿Cómo se podría evitar? (Pista: el reto 2.)
2. ¿`verificar_requisitos` debería ser una herramienta MCP en vez de un agente? ¿Qué ganarías y qué perderías?
3. La tasa vive en `POLITICA_FICTICIA`, que comparten el servidor MCP y el Agente de Créditos porque todo corre en el mismo kernel. En producción, en máquinas distintas, ¿cómo evitarías que cada uno tenga su propia copia de la tasa?
4. ¿Cuántas llamadas a Groq cuesta la pregunta de referencia? Cuenta en la bitácora las delegaciones: cada agente remoto hace al menos dos.

## 🧪 Retos opcionales
1. **Una sola fuente para la tasa.** Haz que el Agente de Créditos obtenga la tasa con `consultar_tasa` por MCP (dale un `MultiServerMCPClient` propio). Así un agente A2A usa MCP por dentro.
2. **Descubrir agentes como se descubren herramientas.** Escribe una función que reciba una lista de URLs, lea cada Agent Card y cree automáticamente una herramienta por agente con su `description` como docstring. Con eso, agregar un tercer agente no tocaría el Asesor.
3. **Un dato que no cabe.** Pregunta con ingresos de 2.000.000 y obligaciones de 300.000 (límite $400.000): ¿qué responde el Asesor? ¿Sigue sin aprobar ni negar?

## ⚠️ Errores frecuentes
| Síntoma | Causa | Qué hacer |
|---|---|---|
| `RuntimeError: El puerto 8001 está ocupado…` | El kernel del notebook 07 sigue vivo con sus servidores | Reinicia o cierra ese kernel y vuelve a ejecutar desde el paso 3 |
| `tools/list` muestra 2 herramientas | Estás hablando con un servidor viejo | Reinicia **este** kernel y ejecuta todo |
| `El puerto … ya lo abrió este notebook` y no ves tus cambios | El servidor de este kernel sigue con el código anterior | Reinicia el kernel y ejecuta todo |
| El Asesor no delega en Requisitos | El LLM decidió no hacerlo | Haz la pregunta más explícita: «verifica los requisitos» |
| Error 401 de Groq | Falta `GROQ_API_KEY` o no diste acceso al notebook | En tu PC: revisa `.env`. En Colab: 🔑 Secrets → activar acceso |
| Error 429 | Muchas llamadas (ahora hay tres agentes con LLM) | Espera unos segundos |

## Fuentes (consultadas el 28-10-2026)
- [Model Context Protocol](https://modelcontextprotocol.io) · [SDK de Python `mcp`](https://pypi.org/project/mcp/)
- [LangChain MCP adapters](https://docs.langchain.com/oss/python/langchain/mcp)
- [Protocolo A2A](https://a2a-protocol.org) · [SDK de Python `a2a-sdk`](https://pypi.org/project/a2a-sdk/)
- [LangChain `create_agent`](https://docs.langchain.com/oss/python/langchain/agents)